In [1]:
import gcsfs
import pandas as pd
from update_vars import GCS_FILE_PATH

## agency vs source_agency
* source_agency refers to the column `agency` that comes from the NTD df itself
* in annual df, this is renamed to `source_agency`
* in monthly df, there is no other join, so it's called `agency`
* in Christian's existing_df, there are 2 columns, `agency` and `source_agency`...why? is this coming from crosswalk?
   * It doesn't look like it's from the join with csv crosswalk either
   * For aggregation, just use `agency` and ignore `source_agency`

In [ ]:
# check the first downloaded df, where is agency vs source_agency from?
agency = pd.read_parquet(
    f"{GCS_FILE_PATH}monthly.parquet",
    filesystem = gcsfs.GCSFileSystem(),
    columns = ["agency"]
).drop_duplicates()

In [ ]:
df = pd.read_parquet(
    f"{GCS_FILE_PATH}monthly_with_crosswalk.parquet",
    filesystem = gcsfs.GCSFileSystem()
)

df[df.agency.str.contains("Monica")][["agency"]].drop_duplicates()

In [ ]:
agency[agency.agency.str.contains("Monica")]

## sanity check, counts by RTPA-year
* be a bit more granular, since monthly has year-month-mode-tos combinations
* if we can narrow it down to the year, we can debug better

In [2]:
EXISTING_GCS = "gs://calitp-analytics-data/data-analyses/ntd/"

MIN_YEAR = 2018
existing_df = pd.read_parquet(
    f"{EXISTING_GCS}ca_monthly_ridership_2026_April.parquet",
    filters = [[("period_year", ">=", MIN_YEAR)]],
    filesystem = gcsfs.GCSFileSystem()
).drop(
    #updated columns names to match new df
    columns = ["mode", "tos"]
).rename(columns = {"Mode_full": "Mode", "TOS_full": "TOS"})

In [3]:
existing_df.dtypes

ntd_id                 object
agency                 object
reporter_type          object
period_year_month      object
period_year             int64
period_month            int64
Status                 object
uza_name               object
upt                   float64
source_agency          object
last_report_year        int64
ntd_id_2022            object
rtpa_name              object
_merge               category
previous_y_m_upt      float64
change_1yr            float64
pct_change_1yr        float64
Mode                   object
TOS                    object
dtype: object

In [4]:
existing2 = existing_df[["agency", "source_agency"]].drop_duplicates()
existing2[existing2.agency.str.contains("Monica")]

,agency,source_agency
2994,City of Santa Monica,City of Santa Monica (BBB) - Department of Tra...


In [5]:
existing_df.rtpa_name.value_counts()

rtpa_name
Metropolitan Transportation Commission                      5725
Los Angeles County Metropolitan Transportation Authority    4214
Sacramento Area Council of Governments                      1147
San Diego Association of Governments                        1109
Riverside County Transportation Commission                   970
Orange County Transportation Authority                       807
San Bernardino County Transportation Authority               792
San Joaquin Council of Governments                           636
Placer County Transportation Planning Agency                 544
Stanislaus Council of Governments                            540
Ventura County Transportation Commission                     501
Tulare County Association of Governments                     500
Transportation Agency for Monterey County                    398
Kings County Association of Governments                      370
Santa Barbara County Association of Governments              347
Santa Cruz Coun

In [6]:
# for monthly, rtpa_name is still rtpa_name_split
# but without the extra shuffling for LACDPW
df = pd.read_parquet(
    f"{GCS_FILE_PATH}monthly_with_crosswalk.parquet",
    filesystem = gcsfs.GCSFileSystem()
).drop(columns = {"rtpa_name"}).rename(
    columns = {"rtpa_name_split": "rtpa_name"}
)

In [7]:
df.month_first_day.nunique(), existing_df.period_year_month.nunique()

(100, 100)

In [8]:
df.month_first_day.min(), df.month_first_day.max()

(Timestamp('2018-01-01 00:00:00'), Timestamp('2026-04-01 00:00:00'))

In [9]:
existing_df.period_year_month.min(), existing_df.period_year_month.max()

('2018-01', '2026-04')

In [10]:
# change this to agency for monthly
def counts_by_rtpa(
    df: pd.DataFrame,
    group_cols: list
) -> pd.DataFrame:
    """
    Use this to read in existing df vs new annual/monthly df
    and do groupby by rtpa_name or rtpa_name_split,
    and see how counts look overall.
    """
    df2 = (
        df
        .groupby(group_cols, dropna=False)
        .agg(
            total_upt=("upt", "sum"),
            n_agencies=("agency", "nunique"),
            agencies=pd.NamedAgg(column="agency", aggfunc=lambda x: list(set(x))),
            ntd_ids=pd.NamedAgg(column="ntd_id", aggfunc=lambda x: list(set(x))),
        ).reset_index()
        .astype({"total_upt": "int64"})
        .sort_values(by=group_cols + ["total_upt"], ascending=False)
        .reset_index(drop=True)
    )

    return df2

In [11]:
# agency, needs to be actually source_agency, however that is derived
df2 = counts_by_rtpa(df, ["year", "rtpa_name"])

In [12]:
existing_df2 = counts_by_rtpa(existing_df, ["period_year", "rtpa_name"]).rename(columns = {"period_year": "year"})

In [13]:
existing_df[existing_df.source_agency.str.contains("Monica")][["agency", "source_agency"]]

,agency,source_agency
2994,City of Santa Monica,City of Santa Monica (BBB) - Department of Tra...
2995,City of Santa Monica,City of Santa Monica (BBB) - Department of Tra...
2996,City of Santa Monica,City of Santa Monica (BBB) - Department of Tra...
2997,City of Santa Monica,City of Santa Monica (BBB) - Department of Tra...
2998,City of Santa Monica,City of Santa Monica (BBB) - Department of Tra...
...,...,...
11924,City of Santa Monica,City of Santa Monica (BBB) - Department of Tra...
11925,City of Santa Monica,City of Santa Monica (BBB) - Department of Tra...
11926,City of Santa Monica,City of Santa Monica (BBB) - Department of Tra...
11927,City of Santa Monica,City of Santa Monica (BBB) - Department of Tra...


In [14]:
compare_df = pd.merge(
    existing_df2, 
    df2,
    on = ["year", "rtpa_name"],
    how = "outer",
    indicator=True
).astype({
    c: "Int64" 
    for c in ["total_upt_x", "total_upt_y", "n_agencies_x", "n_agencies_y"]}
)

In [15]:
compare_df.dtypes

year               int64
rtpa_name         object
total_upt_x        Int64
n_agencies_x       Int64
agencies_x        object
ntd_ids_x         object
total_upt_y        Int64
n_agencies_y       Int64
agencies_y        object
ntd_ids_y         object
_merge          category
dtype: object

In [16]:
compare_df._merge.value_counts()

_merge
both          207
left_only       9
right_only      0
Name: count, dtype: int64

In [17]:
compare_df = compare_df.assign(
    missing_agency = compare_df.apply(
        lambda x:
        list(set([c for c in x.agencies_x if c not in x.agencies_y])) if x._merge=="both"
        else [], 
        axis=1),
    missing_ids = compare_df.apply(
        lambda x:
        list(set([c for c in x.ntd_ids_x if c not in x.ntd_ids_y])) if x._merge=="both"
        else [], 
        axis=1),
    diff_existing_to_new = compare_df.total_upt_x - compare_df.total_upt_y        
)

In [18]:
results = compare_df[(compare_df._merge=="both") & 
    (compare_df.total_upt_x != compare_df.total_upt_y)].reset_index(drop=True)

In [19]:
# same number of agencies in both lists, same ntd_ids, but slightly different upt
# very little differences, but interesting
results

,year,rtpa_name,total_upt_x,n_agencies_x,agencies_x,ntd_ids_x,total_upt_y,n_agencies_y,agencies_y,ntd_ids_y,_merge,missing_agency,missing_ids,diff_existing_to_new
0,2024,Los Angeles County Metropolitan Transportation...,394772968,18,"[City of Glendale, Access Services, City of No...","[90041, 90043, 90157, 90214, 99423, 90171, 900...",394782873,18,"[City of Glendale, Access Services, City of No...","[90041, 90043, 90157, 90214, 99423, 90171, 900...",both,[],[],-9905
1,2025,Los Angeles County Metropolitan Transportation...,390104056,17,"[Access Services, City of Norwalk, City of Red...","[90041, 90043, 90157, 90214, 90171, 90023, 901...",390135428,18,"[City of Glendale, Access Services, City of No...","[90041, 90043, 90157, 90214, 99423, 90171, 900...",both,[],[],-31372
2,2026,Imperial County Transportation Commission,228475,1,[Imperial County Transportation Commission],[90226],228460,1,[Imperial County Transportation Commission],[90226],both,[],[],15
3,2026,Los Angeles County Metropolitan Transportation...,129519004,17,"[Access Services, City of Norwalk, City of Red...","[90041, 90043, 90157, 90214, 90171, 90023, 901...",129530327,18,"[City of Glendale, Access Services, City of No...","[90041, 90043, 90157, 90214, 99423, 90171, 900...",both,[],[],-11323
4,2026,Metropolitan Transportation Commission,118884484,21,"[Peninsula Corridor Joint Powers Board, Napa V...","[90094, 90017, 90013, 90078, 90089, 90225, 900...",118912580,21,"[Peninsula Corridor Joint Powers Board, Napa V...","[90094, 90017, 90013, 90078, 90089, 90225, 900...",both,[],[],-28096
5,2026,Orange County Transportation Authority,14691102,2,"[Anaheim Transportation Network, Orange County...","[90036, 90211]",14690962,2,"[Anaheim Transportation Network, Orange County...","[90036, 90211]",both,[],[],140
6,2026,Santa Cruz County Regional Transportation Comm...,2203930,1,[Santa Cruz Metropolitan Transit District],[90006],2206204,1,[Santa Cruz Metropolitan Transit District],[90006],both,[],[],-2274
7,2026,Tulare County Association of Governments,365370,2,"[Tulare County Regional Transit Agency, City o...","[90310, 90091]",365382,2,"[Tulare County Regional Transit Agency, City o...","[90310, 90091]",both,[],[],-12
8,2026,Ventura County Transportation Commission,1377090,2,"[Ventura County Transportation Commission, Gol...","[90035, 90164]",1355398,2,"[Ventura County Transportation Commission, Gol...","[90035, 90164]",both,[],[],21692


## Differences by year
**interpretation**: missing agency means it appears in existing, but does not appear in newer table

**2024**
* Big Blue Bus missing, only because instead of `City of Santa Monica`, there's `City of Santa Monica (BBB)`

**2025**
* Big Blue Bus missing for same reason

**2026**
* Big Blue Bus same reason
* `Imperial County Transportation Commission (ICTC)` vs `Imperial County Transportation Commission`
* Where are the differences in names coming from?
* Why is `source_agency` from dbt, which should be coming in from `dim_agency_information`, providing different names?
   * Possible, it is dim table. But we really would want 1 name labeled per NTD ID.
   * Is this what we want? We want standardizd names per NTD ID, is that what we're getting?

In [20]:
for rtpa in results.rtpa_name.unique():
    print(rtpa)
    subset_df = results[results.rtpa_name==rtpa].reset_index(drop=True)
    print(subset_df.missing_agency.iloc[0])
    print(subset_df.missing_ids.iloc[0])
    print("******************************************************************")

Los Angeles County Metropolitan Transportation Authority
[]
[]
******************************************************************
Imperial County Transportation Commission
[]
[]
******************************************************************
Metropolitan Transportation Commission
[]
[]
******************************************************************
Orange County Transportation Authority
[]
[]
******************************************************************
Santa Cruz County Regional Transportation Commission
[]
[]
******************************************************************
Tulare County Association of Governments
[]
[]
******************************************************************
Ventura County Transportation Commission
[]
[]
******************************************************************


In [ ]:
# need to figure out which columns need to be kept
monthly_col_dict = {
    "uace_code": "UACE Code",
    #"Dt": "Date",
    "type_of_service": "Type of Service",
    "legacy_ntd_id": "Legacy NTD ID",
    "upt": "UPT",
    "vrm": "VRM",
    "vrh": "VRH",
    "voms": "VOMS",
    "Rtpa": "RTPA", # check which rtpa column to use
    "upt_change_1yr": "Change in 1 Year UPT", # double check this
    "upt_pct_change_1yr": "Percent Change in 1 Year UPT",
    "type_of_service_full_name": "Type of Service Full Name",
}